# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [mlcroissant](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

First, enumerate the record sets, then for each, enumerate the fields available and their `@id`s.

In [ ]:
record_sets = []

print("--- Record Set Overview ---")
for rs in dataset.metadata.record_sets:
    print(f"Record set name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description if hasattr(rs, 'description') else ''}")
    field_summaries = []
    for fld in rs.fields:
        field_summaries.append((fld.name, fld.id, getattr(fld, 'data_type', 'Unknown')))
    print(f"  Fields (")
    for name, fid, dtype in field_summaries:
        print(f"    {name} @id={fid} dataType={dtype}")
    print(")\n")
    record_sets.append(rs.id)

if len(record_sets)==0:
    print("No record sets found in the metadata. Please check dataset definition.")

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
dataframes = {}

for recset_id in record_sets:
    print(f"Loading records for: {recset_id}")
    records = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(records)
    dataframes[recset_id] = df
    print(f"Columns in {recset_id}:", df.columns.tolist())
    display(df.head(3))

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering numeric values, normalizing, grouping, and more. For illustration, we'll try a numeric field that counts peptides as an example.

In [ ]:
# Choose the first record set if available
if record_sets:
    main_record_set = record_sets[0]
    df = dataframes[main_record_set]
    print(f"Sample columns for EDA: {df.columns.tolist()}")

    # Try to find a numeric field automatically by dtype
    candidate_numeric = None
    for col in df.columns:
        col_data = pd.to_numeric(df[col], errors='coerce')
        if col_data.notna().sum() > 0:
            candidate_numeric = col
            break

    if candidate_numeric:
        print(f"Using numeric field for EDA: {candidate_numeric}")
        # Filtering > threshold
        threshold = col_data.mean() if col_data.mean() and not pd.isna(col_data.mean()) else 10
        filtered_df = df[pd.to_numeric(df[candidate_numeric], errors='coerce') > threshold].copy()
        print(f"Filtered records with {candidate_numeric} > {threshold:.2f} (showing first rows):")
        display(filtered_df.head())

        norm_col = f"{candidate_numeric}_normalized"
        filtered_df[norm_col] = (pd.to_numeric(filtered_df[candidate_numeric], errors='coerce') - pd.to_numeric(filtered_df[candidate_numeric], errors='coerce').mean()) / pd.to_numeric(filtered_df[candidate_numeric], errors='coerce').std()
        print(f"Normalized {candidate_numeric} for filtered records:")
        display(filtered_df[[candidate_numeric, norm_col]].head())

        # Try to group by a string category field
        possible_group_fields = [col for col in df.columns if col != candidate_numeric and df[col].dtype == 'object']
        if possible_group_fields:
            group_field = possible_group_fields[0]
            print(f"Grouping by {group_field}")
            grouped = filtered_df.groupby(group_field)[candidate_numeric].mean().reset_index().sort_values(candidate_numeric, ascending=False)
            display(grouped.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found.")
else:
    print("No record sets available for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If a valid numeric and group by field was determined, let's plot
if record_sets and 'candidate_numeric' in locals() and candidate_numeric and filtered_df.shape[0] > 0:
    plt.figure(figsize=(8,5))
    sns.histplot(pd.to_numeric(filtered_df[candidate_numeric], errors='coerce').dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {candidate_numeric}")
    plt.xlabel(candidate_numeric)
    plt.ylabel("Frequency")
    plt.show()

    # If grouped field exists, plot the means
    if 'group_field' in locals():
        top_groups = grouped[grouped[candidate_numeric].notna()].head(10)
        plt.figure(figsize=(8,5))
        sns.barplot(y=top_groups[group_field], x=top_groups[candidate_numeric])
        plt.title(f"Top 10 {group_field} by mean {candidate_numeric}")
        plt.xlabel(f"Mean {candidate_numeric}")
        plt.ylabel(group_field)
        plt.show()
else:
    print("No suitable numeric field for visualization found.")

## 6. Conclusion

In this notebook, we demonstrated how to load a FAIR-compliant dataset described with a Croissant schema using the `mlcroissant` library:  
- We programmatically discovered all record sets and their field `@id`s.  
- We loaded all record sets as dataframes, and performed basic EDA on available numeric columns (including outlier filtering, normalization, and basic grouping).
- We visualized key distributions and group summaries.  

**Next steps:** Data scientists and researchers can further utilize and process the dataset as needed for their own scientific, statistical, or ML workflows. For more details, consult the dataset metadata and [mlcroissant documentation](https://github.com/mlcommons/croissant).